In [0]:
%pip install databricks-sdk mlflow

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
CATALOG = "olympus_hub"  
SCHEMA  = "opsdata"    
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

print(f"Using: {CATALOG}.{SCHEMA}")
print("Config ready ✓")

Using: olympus_hub.opsdata
Config ready ✓


In [0]:
# CELL 3 — Smart AI Diagnosis Engine with pre-processing + post-validation

import json, re
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

# ─────────────────────────────────────────────────────────────
# 1. SIGNAL EXTRACTOR — regex-based pre-processing that pulls
#    structured hints from raw error text BEFORE the LLM sees it
# ─────────────────────────────────────────────────────────────

_HTTP_CODE_MAP = {
    "400": ("BAD_REQUEST", "CONFIG_ERROR"),
    "401": ("UNAUTHORIZED", "AUTH_IDENTITY"),
    "403": ("FORBIDDEN / PERMISSION_DENIED", "PERMISSION_RBAC"),
    "404": ("NOT_FOUND", "CONFIG_ERROR"),
    "408": ("REQUEST_TIMEOUT", "TIMEOUT"),
    "429": ("RATE_LIMITED", "RESOURCE_LIMIT"),
    "500": ("INTERNAL_SERVER_ERROR", "INFRA_FAILURE"),
    "502": ("BAD_GATEWAY", "INFRA_FAILURE"),
    "503": ("SERVICE_UNAVAILABLE", "INFRA_FAILURE"),
    "504": ("GATEWAY_TIMEOUT", "TIMEOUT"),
}

_EXCEPTION_PATTERNS = [
    (r"OutOfMemoryError|GC overhead|Java heap space|oom-killer|memory exhausted", "OOM"),
    (r"TimeoutException|deadline exceeded|timed? ?out|timeout|read timed out", "TIMEOUT"),
    (r"PermissionDenied|Access[_ ]?Denied|Forbidden|Insufficient privileges|not authorized|does not have sufficient permissions|UnauthorizedAccess|AuthorizationPermissionMismatch", "PERMISSION_RBAC"),
    (r"AuthenticationFailed|InvalidClientSecret|AADSTS\d+|expired token|invalid.?token|401 Unauthorized|TokenExpired|SecretNotFound|ClientSecretExpired", "AUTH_IDENTITY"),
    (r"SchemaChange|SCHEMA_MISMATCH|AnalysisException.*cannot resolve|data type mismatch|corrupt|ParseException|BadRecordException|MalformedCSV", "DATA_QUALITY"),
    (r"ClusterNotReady|CLOUD_PROVIDER_LAUNCH_FAILURE|spot.?instance|DRIVER_UNREACHABLE|Cluster.+terminated|ClusterLaunchException", "CLUSTER_FAILURE"),
    (r"ModuleNotFoundError|ImportError|ClassNotFoundException|NoSuchMethodError|version.?conflict|dependency.?resolution|PackageNotFound", "DEPENDENCY"),
    (r"UnknownHostException|ConnectionRefused|DNS.?resolution|NETWORK_ERROR|firewall|PrivateLink|VNet|ssl.?handshake|ConnectTimeout|SocketTimeout", "NETWORK"),
    (r"InvalidConfig|No such file|FileNotFoundException|path does not exist|Wrong.?parameter|InvalidParameter|ResourceNotFound.*config|NoSuchKey", "CONFIG_ERROR"),
    (r"QuotaExceeded|RateLimited|Too many requests|throttl|storage.?full|DiskQuota|LimitExceeded|CapacityExceeded", "RESOURCE_LIMIT"),
    (r"Azure.?outage|region.?unavailable|service.?degradation|ARM.?deployment|ProvisioningState.?Failed|InternalServerError.*azure", "INFRA_FAILURE"),
]

_SERVICE_PATTERNS = [
    (r"dev\.azure\.com|Azure DevOps|ADO|vso\.", "Azure DevOps (ADO)"),
    (r"blob\.core\.windows\.net|abfss?://|ADLS|dfs\.core\.windows\.net", "Azure Data Lake Storage (ADLS)"),
    (r"vault\.azure\.net|Key ?Vault|AKV", "Azure Key Vault"),
    (r"login\.microsoftonline|Azure ?AD|Entra|AADSTS|AAD", "Azure AD / Entra ID"),
    (r"Managed.?Identity|DefaultAzureCredential|MSI|Object.?ID.*[a-f0-9-]{36}", "Managed Identity"),
    (r"Service.?Principal|client_id|client_secret|SPN", "Service Principal"),
    (r"databricks\.net|dbfs:/|Unity.?Catalog|delta\.tables", "Databricks Platform"),
    (r"cosmos\.azure\.com|CosmosDB", "Azure Cosmos DB"),
    (r"database\.windows\.net|SQL.?Server|JDBC.*sqlserver", "Azure SQL"),
    (r"servicebus\.windows\.net|EventHub|eventhub", "Azure Event Hubs / Service Bus"),
    (r"azurecr\.io|Container.?Registry", "Azure Container Registry"),
    (r"api\.powerbi|PowerBI|power.?bi", "Power BI"),
    (r"snowflake|snowflakecomputing\.com", "Snowflake"),
    (r"s3\.amazonaws|aws\.amazon|IAM", "AWS"),
]

def extract_signals(error_message: str, log_snippet: str, raw_payload: str) -> dict:
    """Pre-process all text fields with regex to extract structured hints."""
    combined = f"{error_message or ''} {log_snippet or ''} {raw_payload or ''}"
    hints = {
        "http_codes": [],
        "exceptions": [],
        "services": [],
        "suggested_category": None,
        "termination_code": None,
        "state_message": None,
    }

    # Extract HTTP status codes
    for code in re.findall(r'\b[1-5]\d{2}\b', combined):
        if code in _HTTP_CODE_MAP:
            label, cat = _HTTP_CODE_MAP[code]
            hints["http_codes"].append(f"HTTP {code} ({label})")
            if hints["suggested_category"] is None:
                hints["suggested_category"] = cat

    # Match exception / error patterns
    for pattern, cat in _EXCEPTION_PATTERNS:
        if re.search(pattern, combined, re.IGNORECASE):
            hints["exceptions"].append(cat)
            if hints["suggested_category"] is None:
                hints["suggested_category"] = cat

    # Detect services involved
    for pattern, svc in _SERVICE_PATTERNS:
        if re.search(pattern, combined, re.IGNORECASE):
            hints["services"].append(svc)

    # Parse raw_payload JSON for termination details
    try:
        payload_obj = json.loads(raw_payload) if raw_payload else {}
        td = (payload_obj.get("status", {}).get("termination_details", {})
              or payload_obj.get("state", {}))
        hints["termination_code"] = td.get("code", td.get("result_state"))
        hints["state_message"] = td.get("message", td.get("state_message"))
    except Exception:
        pass

    # Deduplicate
    hints["http_codes"] = list(dict.fromkeys(hints["http_codes"]))
    hints["exceptions"] = list(dict.fromkeys(hints["exceptions"]))
    hints["services"]   = list(dict.fromkeys(hints["services"]))

    return hints


# ─────────────────────────────────────────────────────────────
# 2. THE LLM DIAGNOSIS FUNCTION
# ─────────────────────────────────────────────────────────────

def diagnose_signal(signal_type, job_name, error_message, log_snippet,
                    raw_payload="", workspace_id=""):
    """
    Pre-extracts signals → feeds structured hints + raw text to LLM
    → validates output → returns structured diagnosis dict.
    """
    max_len = 4000
    error_msg_trimmed = (error_message or "")[:max_len]
    log_trimmed       = (log_snippet or "")[:max_len]
    payload_trimmed   = (raw_payload or "")[:max_len]

    # ── Step 1: Pre-extract signals ──
    hints = extract_signals(error_message, log_snippet, raw_payload)

    hints_block = ""
    if any([hints["http_codes"], hints["exceptions"], hints["services"],
            hints["termination_code"]]):
        hints_block = f"""
=== PRE-EXTRACTED SIGNALS (use these as strong hints — do NOT ignore) ===
HTTP Status Codes Detected : {', '.join(hints['http_codes']) or 'None'}
Error/Exception Patterns   : {', '.join(hints['exceptions']) or 'None'}
Services/Resources Involved: {', '.join(hints['services']) or 'None'}
Termination Code           : {hints['termination_code'] or 'N/A'}
State Message              : {hints['state_message'] or 'N/A'}
Suggested Category (from pre-processing): {hints['suggested_category'] or 'N/A'}

IMPORTANT: The pre-extracted signals above are derived from the raw error text.
If they suggest a specific category, you MUST use that category unless you have
a MORE specific diagnosis (never downgrade to UNKNOWN or CODE_BUG when clear signals exist).
"""

    prompt = f"""You are a senior Databricks and Azure incident analyst. Your job is to read the error details below and produce a PRECISE, SPECIFIC diagnosis.

RULE #1: NEVER return "Unknown failure" or vague root causes. Every error message contains clues — extract them.
RULE #2: The cause_category MUST match the actual error. A 403 Forbidden is PERMISSION_RBAC, not CODE_BUG.
RULE #3: The root_cause must name the SPECIFIC service, resource, permission scope, or component that failed.
RULE #4: The suggested_fix must contain ACTIONABLE steps (name exact permissions, scopes, services, commands).

=== SIGNAL ===
Signal Type: {signal_type}
Job Name   : {job_name}
Workspace  : {workspace_id}

=== ERROR MESSAGE ===
{error_msg_trimmed}

=== LOG SNIPPET ===
{log_trimmed}

=== RAW PAYLOAD ===
{payload_trimmed}
{hints_block}
=== CAUSE CATEGORIES (pick the MOST specific match) ===
PERMISSION_RBAC  — 403 errors, missing permissions, insufficient scopes, role/ACL issues, Managed Identity access denied
AUTH_IDENTITY    — 401 errors, expired tokens/secrets, authentication failures, Service Principal / MI auth issues
OOM              — OutOfMemoryError, GC overhead, heap exhaustion, driver/executor memory
TIMEOUT          — Timeouts, deadline exceeded, long-running operations killed
DATA_QUALITY     — Schema mismatch, corrupt data, null violations, parsing errors
CLUSTER_FAILURE  — Cluster launch failure, spot eviction, cloud capacity, node issues
DEPENDENCY       — Missing library, import error, version conflicts, external service down
NETWORK          — DNS failure, connectivity timeout, firewall, VNet/PrivateLink issues
CONFIG_ERROR     — Wrong configuration, invalid paths, bad parameters, missing resources (404)
RESOURCE_LIMIT   — 429 rate limit, quota exceeded, throttled, storage full
INFRA_FAILURE    — Azure outage, region issue, ARM failure, 500/502/503 from cloud services
CODE_BUG         — Logic error in USER code only (not infrastructure/permission/auth issues)
UNKNOWN          — ONLY if error message is completely empty or contains zero diagnostic information

=== FEW-SHOT EXAMPLES ===

Example 1 — Permission error:
Error: "403 Forbidden. Managed Identity does not have permissions on organization"
Correct output: {{"root_cause": "Managed Identity lacks required scopes (vso.build_execute, vso.release_manage) on Azure DevOps org", "cause_category": "PERMISSION_RBAC", "confidence": 0.95, ...}}
WRONG output: {{"root_cause": "Unknown failure", "cause_category": "CODE_BUG", ...}}  ← NEVER do this

Example 2 — OOM:
Error: "java.lang.OutOfMemoryError: Java heap space"
Correct: {{"root_cause": "Driver JVM ran out of heap memory processing large dataset", "cause_category": "OOM", "confidence": 0.95, ...}}

Example 3 — Auth expiry:
Error: "AADSTS7000215: Invalid client secret provided"
Correct: {{"root_cause": "Service Principal client secret has expired or is invalid in Azure AD", "cause_category": "AUTH_IDENTITY", "confidence": 0.95, ...}}

Example 4 — Network:
Error: "UnknownHostException: mystorageaccount.blob.core.windows.net"
Correct: {{"root_cause": "DNS resolution failed for storage account — possible VNet/Private Link misconfiguration", "cause_category": "NETWORK", "confidence": 0.9, ...}}

Example 5 — Rate limiting:
Error: "HTTP 429 Too Many Requests — Azure Resource Manager throttled"
Correct: {{"root_cause": "ARM API rate limit exceeded for subscription", "cause_category": "RESOURCE_LIMIT", "confidence": 0.9, ...}}

Return ONLY valid JSON (no markdown fences, no explanation):
{{{{
  "root_cause": "specific one-sentence diagnosis naming the exact service/resource/permission",
  "cause_category": "CATEGORY_FROM_LIST",
  "confidence": 0.85,
  "suggested_fix": "step-by-step actionable fix with specific commands, permissions, or portal steps",
  "business_impact": {{"sla_at_risk": true/false, "severity": "low/medium/high/critical"}},
  "auto_resolved": false
}}}}"""

    # ── Step 2: Call LLM ──
    response = w.serving_endpoints.query(
        name=LLM_ENDPOINT,
        messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
        temperature=0.05,   # Near-deterministic
        max_tokens=800
    )
    raw_text = response.choices[0].message.content

    # ── Step 3: Parse JSON from LLM ──
    result = _parse_llm_json(raw_text)

    # ── Step 4: Post-validation — override lazy/generic answers ──
    result = _validate_diagnosis(result, hints)

    return result


def _parse_llm_json(raw_text: str) -> dict:
    """Robust JSON extraction from LLM output."""
    # Try direct parse
    try:
        return json.loads(raw_text.strip())
    except Exception:
        pass
    # Strip code fences
    cleaned = re.sub(r'```(?:json)?\s*', '', raw_text).strip().rstrip('`')
    try:
        return json.loads(cleaned)
    except Exception:
        pass
    # Extract first JSON object
    match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', cleaned, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    # Last resort
    return {"root_cause": "Parse error — LLM did not return valid JSON",
            "cause_category": "UNKNOWN", "confidence": 0.0,
            "suggested_fix": "Review raw LLM output",
            "business_impact": {"sla_at_risk": False, "severity": "low"},
            "auto_resolved": False}


def _validate_diagnosis(result: dict, hints: dict) -> dict:
    """Post-validation: catch and fix lazy/generic LLM answers."""
    category = (result.get("cause_category") or "UNKNOWN").upper()
    root_cause = (result.get("root_cause") or "").lower()

    is_generic = (
        category in ("UNKNOWN", "CODE_BUG")
        or "unknown failure" in root_cause
        or "manual investigation" in root_cause
        or (len(root_cause) < 15 and "unknown" in root_cause)
    )

    # If the LLM gave a generic answer but we found clear signals, override
    if is_generic and hints.get("suggested_category"):
        result["cause_category"] = hints["suggested_category"]
        if result.get("confidence", 0) < 0.5:
            result["confidence"] = 0.7
        # Build a better root_cause from hints
        services = ", ".join(hints["services"]) if hints["services"] else "target service"
        codes = ", ".join(hints["http_codes"]) if hints["http_codes"] else ""
        result["root_cause"] = (
            f"Auto-corrected: {hints['suggested_category']} issue detected "
            f"involving {services}. {codes}. "
            f"Termination: {hints.get('termination_code', 'N/A')}"
        )
        result["suggested_fix"] = (
            f"Investigate {hints['suggested_category']} issue with {services}. "
            f"Check permissions, credentials, and connectivity for the involved services."
        )

    # Normalize category casing
    valid_cats = {"PERMISSION_RBAC", "AUTH_IDENTITY", "OOM", "TIMEOUT",
                  "DATA_QUALITY", "CLUSTER_FAILURE", "DEPENDENCY", "NETWORK",
                  "CONFIG_ERROR", "RESOURCE_LIMIT", "INFRA_FAILURE", "CODE_BUG", "UNKNOWN"}
    if result.get("cause_category", "").upper() not in valid_cats:
        # Try to find closest match
        cat_upper = result.get("cause_category", "").upper()
        for v in valid_cats:
            if v in cat_upper or cat_upper in v:
                result["cause_category"] = v
                break
        else:
            if hints.get("suggested_category"):
                result["cause_category"] = hints["suggested_category"]

    return result


# ──────────────────────────────────────────────────
# QUICK TESTS — multiple error types
# ──────────────────────────────────────────────────
test_cases = [
    {
        "label": "ADO Permission Error (should → PERMISSION_RBAC)",
        "signal_type": "job_failure",
        "job_name": "ado_artifact_sync_pipeline",
        "error_message": 'RuntimeError: Pipeline ado_artifact_sync_pipeline FAILED — ADO Managed Identity access denied: Azure DevOps API returned 403 Forbidden. The Managed Identity (Object ID: 9e2c4a8f-1b3d-5e7f-a2c4-6d8e0f1a3b5d) does not have sufficient permissions on organization pandora-platform. Required scope: vso.build_execute, vso.release_manage. Current grants: vso.build (read-only).',
        "log_snippet": "",
        "raw_payload": '{"state": {"life_cycle_state": "INTERNAL_ERROR", "result_state": "FAILED"}, "status": {"termination_details": {"code": "RUN_EXECUTION_ERROR", "type": "CLIENT_ERROR"}}}'
    },
    {
        "label": "OOM Error (should → OOM)",
        "signal_type": "job_failure",
        "job_name": "daily_etl_pipeline",
        "error_message": "java.lang.OutOfMemoryError: GC overhead limit exceeded",
        "log_snippet": "Driver memory 4g, executor memory 2g, 500M rows",
        "raw_payload": '{"state": {"result_state": "FAILED"}, "status": {"termination_details": {"code": "RUN_EXECUTION_ERROR"}}}'
    },
    {
        "label": "Auth Token Expired (should → AUTH_IDENTITY)",
        "signal_type": "job_failure",
        "job_name": "keyvault_refresh_job",
        "error_message": "AADSTS7000215: Invalid client secret provided. Ensure the secret being sent is the client secret value, not the client secret ID.",
        "log_snippet": "",
        "raw_payload": '{"status": {"termination_details": {"code": "RUN_EXECUTION_ERROR"}}}'
    },
]

for tc in test_cases:
    print(f"\n{'='*60}")
    print(f"TEST: {tc['label']}")
    print('='*60)
    r = diagnose_signal(
        signal_type=tc["signal_type"], job_name=tc["job_name"],
        error_message=tc["error_message"], log_snippet=tc["log_snippet"],
        raw_payload=tc["raw_payload"]
    )
    print(f"  Category : {r['cause_category']}")
    print(f"  Root Cause: {r['root_cause']}")
    print(f"  Confidence: {r['confidence']}")
    print(f"  Fix       : {r['suggested_fix'][:120]}")
    print(f"  ✅ PASS" if tc['label'].split('→')[1].strip().rstrip(')') == r['cause_category'] else f"  ❌ EXPECTED {tc['label'].split('→')[1].strip().rstrip(')')}")

print("\n🎯 Diagnosis engine ready.")


TEST: ADO Permission Error (should → PERMISSION_RBAC)
  Category : PERMISSION_RBAC
  Root Cause: Managed Identity (Object ID: 9e2c4a8f-1b3d-5e7f-a2c4-6d8e0f1a3b5d) lacks required scopes (vso.build_execute, vso.release_manage) on Azure DevOps organization pandora-platform
  Confidence: 0.95
  Fix       : Grant the Managed Identity the necessary permissions by adding the vso.build_execute and vso.release_manage scopes to it
  ✅ PASS

TEST: OOM Error (should → OOM)
  Category : OOM
  Root Cause: Driver JVM ran out of heap memory processing large dataset of 500M rows with insufficient memory allocation of 4g for driver and 2g for executor
  Confidence: 0.95
  Fix       : Increase the driver and executor memory allocation in the Databricks cluster configuration, for example, set spark.drive
  ✅ PASS

TEST: Auth Token Expired (should → AUTH_IDENTITY)
  Category : AUTH_IDENTITY
  Root Cause: Service Principal client secret has expired or is invalid in Azure AD for the keyvault_refresh_job
  

In [0]:
# CELL 4 - Reads raw_signals, diagnoses each one, writes to incidents table

from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, FloatType, BooleanType
import uuid

def process_new_signals():
    """
    Reads signals that haven't been diagnosed yet → runs AI → saves to incidents
    """
    
    print("Reading unprocessed signals...")
    
    # Read raw signals that don't have a matching incident yet
    raw_signals = spark.sql(f"""
        SELECT r.*
        FROM {CATALOG}.{SCHEMA}.raw_signals r
        LEFT JOIN {CATALOG}.{SCHEMA}.incidents i 
            ON r.signal_id = i.incident_id
        WHERE i.incident_id IS NULL
        ORDER BY r.signal_id
        LIMIT 100
    """)
    
    count = raw_signals.count()
    print(f"Found {count} unprocessed signals")
    
    if count == 0:
        print("Nothing to process. Done!")
        return
    
    # Convert to Python list so we can call the LLM row by row
    signals_list = raw_signals.toPandas().to_dict(orient="records")
    
    results = []
    
    for i, signal in enumerate(signals_list):
        print(f"Diagnosing signal {i+1}/{count}: {signal.get('job_name', 'unknown job')}")
        
        try:
            diagnosis = diagnose_signal(
                signal_type   = signal.get("signal_type", "UNKNOWN"),
                job_name      = signal.get("job_name", "unknown"),
                error_message = signal.get("error_message", ""),
                log_snippet   = signal.get("log_snippet", ""),
                raw_payload   = signal.get("raw_payload", ""),
                workspace_id  = signal.get("workspace_id", "")
            )
            
            # Handle business_impact - support both old string format and new dict format
            biz_impact = diagnosis.get("business_impact", "low")
            if isinstance(biz_impact, dict):
                biz_impact_str = json.dumps(biz_impact)
            else:
                biz_impact_str = str(biz_impact)
            
            # Build the incident row
            results.append({
                "incident_id"    : str(uuid.uuid4()),
                "workspace_id"   : signal.get("workspace_id", ""),
                "team_name"      : signal.get("rg_name", "unknown"),   # using rg_name as team
                "job_name"       : signal.get("job_name", "unknown"),
                "root_cause"     : diagnosis.get("root_cause", ""),
                "cause_category" : diagnosis.get("cause_category", "UNKNOWN"),
                "confidence"     : float(diagnosis.get("confidence", 0.0)),
                "suggested_fix"  : diagnosis.get("suggested_fix", ""),
                "business_impact": biz_impact_str,
                "auto_resolved"  : bool(diagnosis.get("auto_resolved", False)),
            })
            
        except Exception as e:
            print(f"  ⚠️ Failed on signal {signal.get('signal_id')}: {e}")
            continue
    
    if not results:
        print("No results to write")
        return
    
    # Convert results to a Spark DataFrame
    incidents_df = spark.createDataFrame(results)
    
    # Add timestamps + mttr_minutes default
    incidents_df = incidents_df \
        .withColumn("detected_at",   current_timestamp()) \
        .withColumn("detected_date", col("detected_at").cast("date")) \
        .withColumn("resolved_at",   lit(None).cast("timestamp")) \
        .withColumn("created_at",    current_timestamp()) \
        .withColumn("mttr_minutes",  lit(0.0).cast("double"))   # default 0.0 for unresolved incidents
    
    # Write to the incidents table (MERGE so we don't create duplicates)
    incidents_df.createOrReplaceTempView("new_incidents")
    
    spark.sql(f"""
        MERGE INTO {CATALOG}.{SCHEMA}.incidents AS target
        USING new_incidents AS source
            ON target.job_name = source.job_name 
           AND target.root_cause = source.root_cause
           AND target.detected_date = source.detected_date
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    print(f"\n✅ Done! Wrote {len(results)} incidents to {CATALOG}.{SCHEMA}.incidents")


# RUN IT
process_new_signals()

Reading unprocessed signals...
Found 100 unprocessed signals
Diagnosing signal 1/100: collector_agent_smoke_test
Diagnosing signal 2/100: collector_e2e_fail_3
Diagnosing signal 3/100: collector_agent_smoke_test
Diagnosing signal 4/100: collector_agent_smoke_test
Diagnosing signal 5/100: synapse_warehouse_refresh
Diagnosing signal 6/100: ado_artifact_sync_pipeline
Diagnosing signal 7/100: collector_e2e_fail_2
Diagnosing signal 8/100: collector_e2e_fail_2
Diagnosing signal 9/100: collector_agent_smoke_test
Diagnosing signal 10/100: collector_agent_smoke_test
Diagnosing signal 11/100: etl_daily_sales
Diagnosing signal 12/100: collector_e2e_fail_3
Diagnosing signal 13/100: ado_artifact_sync_pipeline
Diagnosing signal 14/100: collector_agent_smoke_test
Diagnosing signal 15/100: ado_artifact_sync_pipeline
Diagnosing signal 16/100: collector_e2e_fail_1
Diagnosing signal 17/100: collector_agent_smoke_test
Diagnosing signal 18/100: synapse_warehouse_refresh
Diagnosing signal 19/100: collector_a

In [0]:
# Test cell: Validate mttr_minutes default value before updating Cell 4

from pyspark.sql.functions import col, current_timestamp, lit
import uuid

TEST_INCIDENT_ID = f"__test_{uuid.uuid4().hex[:8]}"

# 1. Build a single test row mimicking what Cell 4 produces, now WITH mttr_minutes
test_data = [{
    "incident_id"    : TEST_INCIDENT_ID,
    "workspace_id"   : "test_workspace",
    "team_name"      : "test_team",
    "job_name"       : "__mttr_test_job",
    "root_cause"     : "Test row for mttr_minutes validation",
    "cause_category" : "CONFIG_ERROR",
    "confidence"     : 0.99,
    "suggested_fix"  : "No fix needed — this is a test row",
    "business_impact": '{"sla_at_risk": false, "severity": "low"}',
    "auto_resolved"  : False,
}]

test_df = spark.createDataFrame(test_data)

# Add timestamp columns + mttr_minutes with default 0.0
test_df = test_df \
    .withColumn("detected_at",   current_timestamp()) \
    .withColumn("detected_date", col("detected_at").cast("date")) \
    .withColumn("resolved_at",   lit(None).cast("timestamp")) \
    .withColumn("created_at",    current_timestamp()) \
    .withColumn("mttr_minutes",  lit(0.0).cast("double"))       # <-- NEW column default

# 2. Schema comparison
target_cols = set(c.name for c in spark.table(f"{CATALOG}.{SCHEMA}.incidents").schema)
source_cols = set(test_df.columns)

missing_in_source = target_cols - source_cols
extra_in_source   = source_cols - target_cols

print("=== Schema Validation ===")
print(f"Target table columns : {sorted(target_cols)}")
print(f"Test DataFrame columns: {sorted(source_cols)}")
if missing_in_source:
    print(f"❌ MISSING in source : {missing_in_source}")
else:
    print("✅ All target columns present in source")
if extra_in_source:
    print(f"⚠️  Extra in source  : {extra_in_source}")

# 3. Test MERGE write
test_df.createOrReplaceTempView("test_incident")

spark.sql(f"""
    MERGE INTO {CATALOG}.{SCHEMA}.incidents AS target
    USING test_incident AS source
        ON target.incident_id = source.incident_id
    WHEN NOT MATCHED THEN INSERT *
""")
print("\n✅ MERGE write succeeded with mttr_minutes = 0.0")

# 4. Verify the row landed
verify = spark.sql(f"""
    SELECT incident_id, job_name, mttr_minutes
    FROM {CATALOG}.{SCHEMA}.incidents
    WHERE incident_id = '{TEST_INCIDENT_ID}'
""")
display(verify)

# 5. Cleanup — remove test row
spark.sql(f"""
    DELETE FROM {CATALOG}.{SCHEMA}.incidents
    WHERE incident_id = '{TEST_INCIDENT_ID}'
""")
print(f"🧹 Cleaned up test row: {TEST_INCIDENT_ID}")

=== Schema Validation ===
Target table columns : ['auto_resolved', 'business_impact', 'cause_category', 'confidence', 'created_at', 'detected_at', 'detected_date', 'incident_id', 'job_name', 'mttr_minutes', 'resolved_at', 'root_cause', 'suggested_fix', 'team_name', 'workspace_id']
Test DataFrame columns: ['auto_resolved', 'business_impact', 'cause_category', 'confidence', 'created_at', 'detected_at', 'detected_date', 'incident_id', 'job_name', 'mttr_minutes', 'resolved_at', 'root_cause', 'suggested_fix', 'team_name', 'workspace_id']
✅ All target columns present in source

✅ MERGE write succeeded with mttr_minutes = 0.0


incident_id,job_name,mttr_minutes
__test_5baa374e,__mttr_test_job,0.0


🧹 Cleaned up test row: __test_5baa374e


In [0]:
display(spark.sql(f"""
    SELECT incident_id, job_name, cause_category, confidence,
           business_impact, auto_resolved, detected_at,
           LEFT(root_cause, 80)    AS root_cause,
           LEFT(suggested_fix, 80) AS fix
    FROM {CATALOG}.{SCHEMA}.incidents
    ORDER BY detected_at DESC
    LIMIT 20
"""))

incident_id,job_name,cause_category,confidence,business_impact,auto_resolved,detected_at,root_cause,fix
650f33ca-47ec-4c35-bb98-39b0b1d3b474,collector_agent_smoke_test,CODE_BUG,0.95,"{""sla_at_risk"": false, ""severity"": ""low""}",false,2026-03-20T09:56:23.436Z,"Collector Agent intentionally sent a test signal, indicating a potential issue w",Review the Collector Agent code and the hub ingestion pipeline configuration to
a216967d-2a64-499a-a3b8-6870742d2a49,collector_agent_smoke_test,CODE_BUG,0.95,"{""sla_at_risk"": true, ""severity"": ""medium""}",false,2026-03-20T09:56:23.436Z,"Collector Agent intentionally sent a test signal, indicating a potential issue w",Review the Collector Agent code and the hub ingestion pipeline configuration to
1d8e3f47-c8a4-4164-b4c0-95546bf0d76f,collector_agent_smoke_test,CODE_BUG,0.85,"{""sla_at_risk"": false, ""severity"": ""low""}",false,2026-03-20T09:56:23.436Z,"Collector Agent intentionally sent a test signal, indicating a potential issue w",Review the Collector Agent code and the smoke test configuration to ensure it is
b51d6d60-357b-4944-9b65-7c959598994e,collector_agent_smoke_test,CODE_BUG,0.9,"{""sla_at_risk"": false, ""severity"": ""low""}",false,2026-03-20T09:56:23.436Z,"Collector Agent intentionally sent a test signal, indicating a potential issue w",Review the Collector Agent code and the hub ingestion pipeline configuration to
4f2a853b-1dd9-407a-9609-f6d1fb40dfff,collector_e2e_fail_3,TIMEOUT,0.95,"{""sla_at_risk"": false, ""severity"": ""low""}",false,2026-03-20T09:56:23.436Z,Auto-corrected: TIMEOUT issue detected involving Databricks Platform. . Terminat,"Investigate TIMEOUT issue with Databricks Platform. Check permissions, credentia"
834052e8-5715-404e-b170-4f04205573da,collector_agent_smoke_test,CODE_BUG,0.9,"{""sla_at_risk"": true, ""severity"": ""medium""}",false,2026-03-20T09:56:23.436Z,"Collector Agent intentionally sent a test signal, indicating a potential issue w",Review the Collector Agent code and the hub ingestion pipeline configuration to
0aa27fc6-d16b-4cdc-9b60-6bfd45d0d006,collector_agent_smoke_test,CODE_BUG,0.85,"{""sla_at_risk"": false, ""severity"": ""low""}",false,2026-03-20T09:56:23.436Z,"Collector Agent intentionally sent a test signal, indicating a potential issue w",Review the Collector Agent code and the smoke test configuration to ensure it is
2db3186a-96b5-4815-8489-54744720dfed,collector_agent_smoke_test,CODE_BUG,0.95,"{""sla_at_risk"": true, ""severity"": ""medium""}",false,2026-03-20T09:56:23.436Z,"Collector Agent intentionally sent a test signal, indicating a potential issue w",Review the Collector Agent code and the smoke test configuration to ensure that
5e9d3b12-8464-45ab-ada9-32a79f624a49,collector_agent_smoke_test,CODE_BUG,0.9,"{""sla_at_risk"": false, ""severity"": ""low""}",false,2026-03-20T09:56:23.436Z,"Collector Agent intentionally sent a test signal, indicating a potential issue w",Review the Collector Agent code and the smoke test configuration to ensure it is
c2968abc-4c26-49d6-af3f-a6980c8052c9,collector_e2e_fail_1,TIMEOUT,0.95,"{""sla_at_risk"": false, ""severity"": ""low""}",false,2026-03-20T09:56:23.436Z,Auto-corrected: TIMEOUT issue detected involving Databricks Platform. . Terminat,"Investigate TIMEOUT issue with Databricks Platform. Check permissions, credentia"


In [0]:
# CELL 6 — Metric Insights Engine
# Scans cluster_metrics for threshold breaches → LLM diagnoses each → writes to metric_insights

import json, uuid
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType
)

# ─────────────────────────────────────────────────────────
# 1. THRESHOLD CONFIGURATION
# ─────────────────────────────────────────────────────────

METRIC_THRESHOLDS = {
    # metric_name             : (lower, upper, anomaly_high,       anomaly_low)
    "cpu_utilization_pct":      (15.0,  85.0, "HIGH_CPU",           "UNDER_UTILIZED"),
    "task_completion_rate":     (90.0,  None, None,                 "LOW_TASK_COMPLETION"),
    "driver_memory_used_pct":   (None,  80.0, "MEMORY_PRESSURE",    None),
    "executor_memory_used_pct": (None,  85.0, "MEMORY_PRESSURE",    None),
    "gc_time_pct":              (None,  10.0, "GC_OVERHEAD",        None),
    "disk_used_gb":             (None, 150.0, "STORAGE_SATURATION", None),
    "shuffle_write_gb":         (None,  80.0, "DISK_SPILL",         None),
    "spill_to_disk_gb":         (None,  20.0, "DISK_SPILL",         None),
    "network_latency_ms":       (None,  15.0, "NETWORK_LATENCY",    None),
}


def _detect_breaches_sql():
    """Build SQL to find metric rows breaching thresholds, excluding already-diagnosed."""
    conditions = []
    for metric, (lo, hi, anom_hi, anom_lo) in METRIC_THRESHOLDS.items():
        parts = []
        if hi is not None and anom_hi:
            parts.append(f"(m.metric_name = '{metric}' AND m.metric_value > {hi})")
        if lo is not None and anom_lo:
            parts.append(f"(m.metric_name = '{metric}' AND m.metric_value < {lo})")
        if parts:
            conditions.append(" OR ".join(parts))
    where = " OR ".join(f"({c})" for c in conditions)

    return f"""
        SELECT m.*
        FROM {CATALOG}.{SCHEMA}.cluster_metrics m
        LEFT JOIN {CATALOG}.{SCHEMA}.metric_insights mi
            ON  m.workspace_id     = mi.workspace_id
            AND m.cluster_id       = mi.cluster_id
            AND m.metric_name      = mi.metric_name
            AND m.aggregation_date = mi.aggregation_date
        WHERE mi.insight_id IS NULL
          AND ({where})
        ORDER BY m.workspace_id, m.cluster_name, m.metric_name
        LIMIT 200
    """


# ─────────────────────────────────────────────────────────
# 2. LLM-BASED METRIC DIAGNOSIS
# ─────────────────────────────────────────────────────────

def diagnose_metric(cluster_name, metric_category, metric_name, metric_value,
                    metric_unit, threshold_value, breach_direction, anomaly_type):
    """Ask the LLM to diagnose a metric anomaly with infra-specific context."""

    prompt = f"""You are a senior Databricks infrastructure performance analyst. A cluster metric has breached its threshold. Diagnose the root cause and provide a specific, actionable fix.

RULES:
- Be SPECIFIC: name the exact Spark config, cluster setting, or architectural change needed.
- The root_cause must explain WHY this metric is abnormal (not just restate the metric).
- The suggested_fix must include exact Spark configs, cluster sizing, or Databricks features.
- The cause_category must be one of: RESOURCE_LIMIT, CONFIG_ERROR, WORKLOAD_SKEW, DATA_VOLUME, CLUSTER_SIZING, DRIVER_BOTTLENECK, NETWORK, UNKNOWN.

=== METRIC BREACH ===
Cluster          : {cluster_name}
Metric Category  : {metric_category}
Metric           : {metric_name} = {metric_value} {metric_unit}
Threshold        : {threshold_value} {metric_unit}
Breach           : {breach_direction} (value is {'above' if breach_direction == 'UPPER_BOUND' else 'below'} the limit)
Anomaly Type     : {anomaly_type}

=== ANOMALY CONTEXT ===
HIGH_CPU            → concurrent tasks, skewed partitions, unoptimized joins
MEMORY_PRESSURE     → broadcast too large, collect() on big data, cache overuse
GC_OVERHEAD         → driver heap too small, large broadcast variables
DISK_SPILL          → insufficient memory forcing shuffle to disk, bad partitioning
STORAGE_SATURATION  → temp files not cleaned, checkpoints accumulated
NETWORK_LATENCY     → cross-region reads, storage throttling, VNet issues
LOW_TASK_COMPLETION → executor failures, data skew causing stragglers
UNDER_UTILIZED      → over-provisioned, autoscaling not configured

=== FEW-SHOT EXAMPLES ===

Example 1 — HIGH_CPU 94%:
{{"root_cause": "Concurrent shuffle-heavy ETL jobs competing for cores — sort-merge join on 500M rows", "cause_category": "WORKLOAD_SKEW", "severity": "high", "confidence": 0.92, "suggested_fix": "1) spark.sql.adaptive.enabled=true 2) Partition on date 3) Scale 4→8 workers", "sla_at_risk": true, "auto_resolved": false}}

Example 2 — GC_OVERHEAD 14%:
{{"root_cause": "Driver broadcasting 2GB lookup and collecting large artifacts — GC thrashing on 4g heap", "cause_category": "DRIVER_BOTTLENECK", "severity": "medium", "confidence": 0.88, "suggested_fix": "1) spark.driver.memory=16g 2) spark.sql.autoBroadcastJoinThreshold=50MB", "sla_at_risk": false, "auto_resolved": false}}

Example 3 — UNDER_UTILIZED 12%:
{{"root_cause": "8-worker cluster for adhoc queries but avg utilization under 15% — fits on 2-3 nodes", "cause_category": "CLUSTER_SIZING", "severity": "low", "confidence": 0.85, "suggested_fix": "1) Autoscaling min=2 max=8 2) Switch to DS3_v2 3) Save ~$1200/mo", "sla_at_risk": false, "auto_resolved": false}}

Return ONLY valid JSON:
{{{{
  "root_cause": "specific diagnosis explaining WHY",
  "cause_category": "CATEGORY",
  "severity": "low/medium/high/critical",
  "confidence": 0.85,
  "suggested_fix": "numbered steps with exact Spark configs",
  "sla_at_risk": true/false,
  "auto_resolved": false
}}}}"""

    response = w.serving_endpoints.query(
        name=LLM_ENDPOINT,
        messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
        temperature=0.05,
        max_tokens=600
    )
    return _parse_llm_json(response.choices[0].message.content)


# ─────────────────────────────────────────────────────────
# 3. EXPLICIT SCHEMA (avoids type inference failures on None)
# ─────────────────────────────────────────────────────────

_INSIGHTS_SCHEMA = StructType([
    StructField("insight_id",          StringType(),  True),
    StructField("workspace_id",        StringType(),  False),
    StructField("cluster_id",          StringType(),  True),
    StructField("cluster_name",        StringType(),  True),
    StructField("metric_category",     StringType(),  True),
    StructField("metric_name",         StringType(),  True),
    StructField("metric_value",        DoubleType(),  True),
    StructField("metric_unit",         StringType(),  True),
    StructField("threshold_value",     DoubleType(),  True),
    StructField("threshold_type",      StringType(),  True),
    StructField("anomaly_type",        StringType(),  True),
    StructField("severity",            StringType(),  True),
    StructField("root_cause",          StringType(),  True),
    StructField("cause_category",      StringType(),  True),
    StructField("confidence",          DoubleType(),  True),
    StructField("suggested_fix",       StringType(),  True),
    StructField("business_impact",     StringType(),  True),
    StructField("related_job_names",   StringType(),  True),
    StructField("related_incident_id", StringType(),  True),
    StructField("auto_resolved",       BooleanType(), True),
])


# ─────────────────────────────────────────────────────────
# 4. PROCESS BREACHES → DIAGNOSE → WRITE
# ─────────────────────────────────────────────────────────

def process_metric_insights():
    """Scan cluster_metrics → find threshold breaches → LLM diagnose → write metric_insights."""

    print("Scanning cluster_metrics for threshold breaches...")
    breaches_df = spark.sql(_detect_breaches_sql())
    count = breaches_df.count()
    print(f"Found {count} undiagnosed metric breaches")

    if count == 0:
        print("No breaches to process. Done!")
        return

    breaches = breaches_df.toPandas().to_dict(orient="records")
    results = []

    for i, row in enumerate(breaches):
        metric = row["metric_name"]
        value  = row["metric_value"]
        cfg    = METRIC_THRESHOLDS.get(metric, (None, None, None, None))
        lo, hi, anom_hi, anom_lo = cfg

        if hi is not None and value > hi:
            thresh_val   = hi
            thresh_type  = "UPPER_BOUND"
            anomaly_type = anom_hi or "UNKNOWN"
        elif lo is not None and value < lo:
            thresh_val   = lo
            thresh_type  = "LOWER_BOUND"
            anomaly_type = anom_lo or "UNKNOWN"
        else:
            continue

        print(f"  [{i+1}/{count}] {row['cluster_name']} | {metric} = {value} {row['metric_unit']} "
              f"(threshold: {thresh_val}, {anomaly_type})")

        try:
            dx = diagnose_metric(
                cluster_name    = row["cluster_name"],
                metric_category = row["metric_category"],
                metric_name     = metric,
                metric_value    = value,
                metric_unit     = row["metric_unit"],
                threshold_value = thresh_val,
                breach_direction= thresh_type,
                anomaly_type    = anomaly_type,
            )

            biz_impact = json.dumps({
                "sla_at_risk": dx.get("sla_at_risk", False),
                "affected_jobs": "",
                "estimated_delay_min": 0
            })

            # Use tuples with explicit schema to avoid inference issues
            results.append((
                str(uuid.uuid4()),                          # insight_id
                str(row["workspace_id"]),                   # workspace_id
                str(row["cluster_id"]),                     # cluster_id
                str(row["cluster_name"]),                   # cluster_name
                str(row["metric_category"]),                # metric_category
                str(metric),                                # metric_name
                float(value),                               # metric_value
                str(row["metric_unit"]),                    # metric_unit
                float(thresh_val),                          # threshold_value
                str(thresh_type),                           # threshold_type
                str(anomaly_type),                          # anomaly_type
                str(dx.get("severity", "medium")),          # severity
                str(dx.get("root_cause", "")),              # root_cause
                str(dx.get("cause_category", "UNKNOWN")),   # cause_category
                float(dx.get("confidence", 0.5)),           # confidence
                str(dx.get("suggested_fix", "")),           # suggested_fix
                str(biz_impact),                            # business_impact
                "",                                         # related_job_names
                "",                                         # related_incident_id
                bool(dx.get("auto_resolved", False)),       # auto_resolved
            ))

        except Exception as e:
            print(f"    ⚠️ Failed: {e}")
            continue

    if not results:
        print("No results to write.")
        return

    # Build DataFrame with explicit schema
    insights_df = spark.createDataFrame(results, schema=_INSIGHTS_SCHEMA)
    insights_df = insights_df \
        .withColumn("detected_at",      current_timestamp()) \
        .withColumn("detected_date",    col("detected_at").cast("date")) \
        .withColumn("resolved_at",      lit(None).cast("timestamp")) \
        .withColumn("mttr_minutes",     lit(None).cast("double")) \
        .withColumn("aggregation_date", col("detected_at").cast("date")) \
        .withColumn("created_at",       current_timestamp())

    # MERGE to avoid duplicates
    insights_df.createOrReplaceTempView("new_insights")
    spark.sql(f"""
        MERGE INTO {CATALOG}.{SCHEMA}.metric_insights AS target
        USING new_insights AS source
            ON  target.workspace_id     = source.workspace_id
            AND target.cluster_id       = source.cluster_id
            AND target.metric_name      = source.metric_name
            AND target.aggregation_date = source.aggregation_date
        WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"\n✅ Done! Wrote {len(results)} metric insights to {CATALOG}.{SCHEMA}.metric_insights")


# ── RUN IT ──
process_metric_insights()

Scanning cluster_metrics for threshold breaches...
Found 200 undiagnosed metric breaches
  [1/200] adhoc-analytics-cluster | disk_used_gb = 161.45 GB (threshold: 150.0, STORAGE_SATURATION)
  [2/200] adhoc-analytics-cluster | disk_used_gb = 163.0 GB (threshold: 150.0, STORAGE_SATURATION)
  [3/200] adhoc-analytics-cluster | disk_used_gb = 184.37 GB (threshold: 150.0, STORAGE_SATURATION)
  [4/200] adhoc-analytics-cluster | disk_used_gb = 174.56 GB (threshold: 150.0, STORAGE_SATURATION)
  [5/200] adhoc-analytics-cluster | driver_memory_used_pct = 83.79 percent (threshold: 80.0, MEMORY_PRESSURE)
  [6/200] adhoc-analytics-cluster | gc_time_pct = 12.81 percent (threshold: 10.0, GC_OVERHEAD)
  [7/200] adhoc-analytics-cluster | gc_time_pct = 12.7 percent (threshold: 10.0, GC_OVERHEAD)
  [8/200] adhoc-analytics-cluster | network_latency_ms = 21.33 ms (threshold: 15.0, NETWORK_LATENCY)
  [9/200] adhoc-analytics-cluster | network_latency_ms = 19.5 ms (threshold: 15.0, NETWORK_LATENCY)
  [10/200] a

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-6290260010712740>, line 265
    261     print(f"\n✅ Done! Wrote {len(results)} metric insights to {CATALOG}.{SCHEMA}.metric_insights")
    264 # ── RUN IT ──
--> 265 process_metric_insights()

File <command-6290260010712740>, line 251, in process_metric_insights()
    249 # MERGE to avoid duplicates
    250 insights_df.createOrReplaceTempView("new_insights")
--> 251 spark.sql(f"""
    252     MERGE INTO {CATALOG}.{SCHEMA}.metric_insights AS target
    253     USING new_insights AS source
    254         ON  target.workspace_id     = source.workspace_id
    255         AND target.cluster_id       = source.cluster_id
    256         AND target.metric_name      = source.metric_name
    257         AND target.aggregation_date = source.aggregation_date
    258     WHEN NOT MATCHED THEN INSERT *
    259 """)
    261 print(f"\n✅ 